# 10. Speech-to-Text — Batch Transcription of an Audio File (Azure AI Speech)

This notebook transcribes a pre-recorded audio file (`conversation.wav`) to text using a **batch/file-based
transcription client** (`azure.ai.transcription.TranscriptionClient`). This is a different Python package from
the classic Speech SDK (`azure-cognitiveservices-speech`) used in notebooks 11–13 — it's a newer, narrower
client focused specifically on transcribing existing audio files rather than live microphone streams.

**Difficulty:** Beginner


## Prerequisites

**Python packages:**
- `azure-ai-transcription` — **not yet in the repo's root `requirements.txt`**, install with:
  ```
  pip3 install azure-ai-transcription
  ```
  This is a distinct package/preview client from `azure-cognitiveservices-speech` (used in notebooks 11–13) —
  don't confuse the two when installing.
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Speech resource with a key and endpoint
- An audio file to transcribe — the original script uses `conversation.wav`, present alongside these scripts
  in this folder

**Environment variables** (add to root `.env`):
```
AZURE_SPEECH_ENDPOINT=https://<your-speech-resource>.cognitiveservices.azure.com/
AZURE_SPEECH_KEY=<your-speech-resource-key>
```


## What You'll Learn

- How to transcribe a pre-recorded audio file (batch/file-based transcription) rather than a live microphone
  stream
- The difference between file-based and real-time/streaming speech recognition (compare to notebook 11)
- How `TranscriptionOptions(locales=[...])` configures the recognition locale for a transcription job
- Where `result.combined_phrases` fits into the response shape


### Step 1 — Create the transcription client

`TranscriptionClient(endpoint, credential)` follows the same `AzureKeyCredential` pattern used by the Language
SDK notebooks (05–07) — auth is key-based, pointed at your Speech resource's Cognitive Services endpoint.

💡 **Exam tip:** Azure AI Speech offers two broad recognition modes worth distinguishing for AI-102:
**real-time/streaming recognition** (low latency, processes audio as it arrives — notebook 11) and
**batch/file transcription** (submit a complete audio file, optimized for throughput/accuracy over latency —
this notebook). Choose batch transcription for offline processing of recordings (e.g. call-center archives),
and real-time for live scenarios (e.g. live captioning, voice assistants).

🔄 **Alternatives:** The classic Speech SDK (`azure-cognitiveservices-speech`, notebooks 11–13) can also
transcribe a file by pointing `AudioConfig(filename=...)` at a WAV file and running `SpeechRecognizer` in
single-shot or continuous mode — useful if you want one consistent SDK/client across both file-based and
live-microphone scenarios instead of two separate packages.


In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.transcription import TranscriptionClient
from azure.core.credentials import AzureKeyCredential
from azure.ai.transcription.models import TranscriptionContent, TranscriptionOptions

load_dotenv()

endpoint = os.environ["AZURE_SPEECH_ENDPOINT"]
api_key = os.environ["AZURE_SPEECH_KEY"]

client = TranscriptionClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key)
)

### Step 2 — Transcribe the audio file

`TranscriptionOptions(locales=["en-US"])` tells the service which language/locale to expect in the audio —
similar in spirit to the `language="en"` argument in the Language SDK calls (notebooks 05, 07), but here it's
the *spoken* language being recognized, not the language of already-transcribed text.

The audio file is opened in binary mode (`"rb"`) and wrapped, along with the options, in a
`TranscriptionContent` object passed to `client.transcribe(...)`.

💡 **Exam tip:** Locale strings for Speech follow the `<language>-<region>` format (e.g. `en-US`, `en-GB`,
`fr-FR`, `ja-JP`) — the region part matters because pronunciation/vocabulary differs by region even for the
same language, and AI-102 expects you to recognize this format across both Speech-to-Text and Text-to-Speech
(notebook 12) configuration.

🔄 **Alternatives:** Pass multiple candidate locales for auto language identification if you don't know the
spoken language ahead of time (check the SDK's language-identification options), instead of hardcoding a
single locale as this script does.


In [ ]:
audio_path = "conversation.wav"
with open(audio_path, "rb") as audio_file:
    options = TranscriptionOptions(locales=["en-US"])
    result = client.transcribe(TranscriptionContent(definition=options, audio=audio_file))

### Step 3 — Print the transcribed text

`result.combined_phrases` holds the full transcription, reconstructed from the individually recognized speech
segments/phrases into one continuous text per speaker-independent pass — `[0].text` is the first (here, only)
combined transcript.

💡 **Exam tip:** Batch transcription results are typically richer than just plain text — depending on the
options requested, they can include per-word timestamps, speaker diarization (who said what), and confidence
scores. This script only prints the combined text, but know that the underlying result object usually carries
more structure for production transcript-processing pipelines.

🔄 **Alternatives:** If you need per-speaker attribution (diarization) or word-level timing, look for
diarization/timestamp options on `TranscriptionOptions` rather than reading only `combined_phrases`.


In [ ]:
print(result.combined_phrases[0].text)

## Summary

This notebook transcribed a pre-recorded WAV file to text using Azure AI Speech's batch transcription client,
configured for the `en-US` locale, and printed the combined transcript. This file-based, non-streaming
approach is the right fit for offline processing of existing recordings — contrast with notebook 11's
real-time, microphone-driven continuous recognition.


## Try It Yourself

1. **Easy:** Point `audio_path` at a different WAV file (e.g. `cloudxeus_support_message.wav`, generated by
   notebook 12) and rerun.
2. **Intermediate:** Change `locales` to a non-English locale that doesn't match the audio's actual language,
   and observe how the transcription quality degrades — reinforcing why locale selection matters.
3. **Advanced:** Loop over all `.wav` files in this folder, transcribe each one, and print a
   `{filename: transcript}` summary — handling any per-file errors so one bad file doesn't stop the batch.
